In [1]:
import optuna
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
# data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
# data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [3]:
def split_datasets_intersection(ace, disc, split_date, scenario):
    a = ace.copy()
    d = disc.copy()

    # Выравниваем временные индексы так, чтобы все наборы были на одинаковых данных в пересечениях
    common_idx = a.index.intersection(d.index)
    a = a.loc[common_idx].dropna(axis=0, how='any')
    d = d.loc[common_idx].dropna(axis=0, how='any')

    # Дата разделения
    split_date = pd.Timestamp(split_date)

    # Сценарии
    if scenario == 'ace-ace':
        train = a[a.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'ace-disc':
        train = a[a.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    elif scenario == 'disc-ace':
        train = d[d.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'disc-disc':
        train = d[d.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    else:
        raise ValueError(f"Некорректный сценарий '{scenario}'. "
                         f"Должен быть один из: 'ace-ace', 'ace-disc', 'disc-ace', 'disc-disc'")

    return train, test

In [9]:
def objective(trial):
    train, test = split_datasets_intersection(
        data_ace_24,
        data_discover_24,
        split_date='2022-01-01',
        scenario='disc-disc'
    )

    target = 'Dst_plus1'
    future_lags = [f'Dst_plus{i}' for i in range(1, 25)]

    feature_cols = [c for c in train.columns if c not in future_lags]

    train = train.dropna(subset=feature_cols + [target])
    test  = test.dropna(subset=feature_cols + [target])

    X_train, y_train = train[feature_cols].values, train[target].values
    X_test,  y_test  = test[feature_cols].values,  test[target].values

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 3, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),

        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),

        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),

        "random_state": 42,
        "metric": "rmse",
        # "verbosity": -1
    }

    model = LGBMRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse"
    )

    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds)

    return rmse

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=50,     # увеличь при необходимости
    n_jobs=-1
)

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


best_model = LGBMRegressor(
    **study.best_params,
    random_state=42,
    metric="rmse"
)

train, test = split_datasets_intersection(
    data_ace_24,
    data_discover_24,
    split_date="2022-01-01",
    scenario="disc-disc"
)

target = 'Dst_plus1'
future_lags = [f'Dst_plus{i}' for i in range(1, 25)]

feature_cols = [c for c in train.columns if c not in future_lags]

train = train.dropna(subset=feature_cols + [target])

X_train, y_train = train[feature_cols].values, train[target].values

best_model.fit(X_train, y_train)

[I 2026-03-26 16:04:31,368] A new study created in memory with name: no-name-39368453-7b02-426f-be91-4653efe9bd1b


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010499 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48050
[LightGBM] [Info] Number of data points in the train set: 41246, number of used features: 200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009748 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48050
[LightGBM] [Info] Number of data points in the train set: 41246, number of used features: 200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009782 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48050
[LightGBM] [Info] Number of data points in the train set: 41246, number of used features: 200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008478 seconds.
You can set `force_col_

[I 2026-03-26 16:05:05,234] Trial 0 finished with value: 20.10148618591399 and parameters: {'n_estimators': 1461, 'learning_rate': 0.015917724804307903, 'num_leaves': 3, 'max_depth': 9, 'subsample': 0.9988449921684028, 'colsample_bytree': 0.595121346265279, 'reg_alpha': 0.00362417522130555, 'reg_lambda': 2.37626174919032, 'min_child_samples': 33}. Best is trial 0 with value: 20.10148618591399.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-03-26 16:05:11,102] Trial 8 finished with value: 20.194576354924067 and parameters: {'n_estimators': 1162, 'learning_rate': 0.010657600745924898, 'num_leaves': 57, 'max_depth': 3, 'subsample': 0.7247424958664201, 'colsample_bytree': 0.5346512967531029, 'reg_alpha': 0.002109419749711268, 'reg_lambda': 0.0034511054365595098, 'min_child_samples': 40}. Best is trial 0 with value: 20.10148618591399.


...

[I 2026-03-26 16:16:40,517] Trial 45 finished with value: 17.097538815226066 and parameters: {'n_estimators': 1391, 'learning_rate': 0.02453732844477745, 'num_leaves': 94, 'max_depth': 7, 'subsample': 0.68397134676046, 'colsample_bytree': 0.9639994375975324, 'reg_alpha': 0.006248306564361479, 'reg_lambda': 0.002343211422870169, 'min_child_samples': 26}. Best is trial 49 with value: 16.431483512448597.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-03-26 16:16:44,120] Trial 44 finished with value: 16.91987991609041 and parameters: {'n_estimators': 1413, 'learning_rate': 0.0222415027774205, 'num_leaves': 97, 'max_depth': 7, 'subsample': 0.7108095760995827, 'colsample_bytree': 0.9774412109182516, 'reg_alpha': 0.004470186184252529, 'reg_lambda': 0.0065314104674755, 'min_child_samples': 27}. Best is trial 49 with value: 16.431483512448597.


Best RMSE: 16.431483512448597
Best params: {'n_estimators': 1160, 'learning_rate': 0.014357868416776678, 'num_leaves': 93, 'max_depth': 5, 'subsample': 0.5606239658637814, 'colsample_bytree': 0.9879825765791656, 'reg_alpha': 0.022687699993507067, 'reg_lambda': 0.005915305250069859, 'min_child_samples': 25}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008050 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48050
[LightGBM] [Info] Number of data points in the train set: 41246, number of used features: 200
[LightGBM] [Info] Start training from score -6.744606
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

,boosting_type,'gbdt'
,num_leaves,93
,max_depth,5
,learning_rate,0.014357868416776678
,n_estimators,1160
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,25
